# Scam Detector: run everything (Person 1)
Run the cells **top to bottom**, one at a time (play button or Shift+Enter).

**Before you start:** Runtime > Change runtime type > **T4 GPU** > Save.

What this notebook does: installs tools, loads your project zip, trains both text models, tests everything
(models, OCR, full pipeline), saves the results to Google Drive, and builds the folder you upload to
Hugging Face to put the app online. Total time: about 15 minutes.

## 1. Install tools

In [ ]:
!nvidia-smi -L || echo "NO GPU FOUND: Runtime > Change runtime type > T4 GPU, then run this cell again"
!pip install -q transformers opencv-python-headless pytesseract pandas scikit-learn joblib streamlit
!apt-get -qq install tesseract-ocr > /dev/null
print("tools installed")

## 2. Upload the project zip
Click **Choose files** and select `scam-detector-person1-complete.zip`.

In [ ]:
from google.colab import files
import zipfile, os, shutil
up = files.upload()
name = next(iter(up))
shutil.rmtree("/content/scam-detector", ignore_errors=True)
zipfile.ZipFile(name).extractall("/content/scam-detector")
%cd /content/scam-detector
!ls

## 3. Prepare data and train both models (about 3 minutes)

In [ ]:
!python ocr/make_synthetic_screenshots.py
!python nlp/prepare_data.py
!python nlp/train_tfidf.py
!python nlp/train_distilbert.py

## 4. Test everything
Copy this whole output into your notes for the report. The **hard-set table** and the **link probe** are the most important results.

In [ ]:
!python nlp/evaluate_indian.py
!python nlp/evaluate_hard.py
!python nlp/probe_shortcuts.py

In [ ]:
!python ocr/evaluate_ocr.py --engine tesseract --modes none,dual
!python nlp/evaluate_pipeline.py
!python nlp/evaluate_pipeline.py --model ensemble

## 5. Save models and reports to Google Drive (so you never lose them)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/scam_detector_models
!cp -r models reports /content/drive/MyDrive/scam_detector_models/
print("saved to MyDrive/scam_detector_models")

## 6. Build the online-deployment bundle
Creates `space_bundle.zip` (code + trained models + settings for Hugging Face Spaces).

In [ ]:
!python deploy/make_space_bundle.py
!cp space_bundle.zip /content/drive/MyDrive/scam_detector_models/
print("Copied to Google Drive: MyDrive/scam_detector_models/space_bundle.zip")

## 7. (Optional) Try the app right now from Colab
Prints a temporary public link that works while this notebook is running.

In [ ]:
import subprocess, time, re, os
os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared")
subprocess.Popen(["streamlit", "run", "frontend/app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(10)
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                       stderr=subprocess.PIPE, text=True)
for line in tun.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        print("Open this link:", m.group(0))
        break